# 08 - Pemantauan model dan kebijakan (proposal §6.4)
Proposal meminta: *'PSI pada input dan skor, KS dan Gini pada hasil yang sudah matang, tier-migration matrix, distribusi reason code'*, uji fairness per wilayah, gender, dan usia, serta pemicu retrain: **PSI skor > 0,25 atau KS turun > 5 poin**.

Notebook ini membangun pemantau (`monitoring/monitor.py`) dan **mengujinya**, karena pemantau yang belum terbukti bisa menyala pada hal yang benar adalah pemantau yang tidak bisa dipercaya:
* **Uji alarm palsu:** pada data yang sah stabil (DEV Juli-September, dan nasabah simulasi baru), pemantau tidak boleh menyala berlebihan.
* **Uji deteksi:** pada data yang sengaja dirusak (sumber payee mati, perilaku bergeser seperti Lebaran), pemantau harus menyala dan menunjuk fitur yang benar.
* Lalu pemantauan kinerja bulanan, fairness per kuartal, tier migration dan churn, serta distribusi reason code.

Ambang PSI (konvensi industri): < 0,10 stabil; 0,10-0,25 pantau; > 0,25 drift.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")      # notebooks use paths relative to the project root
print("working directory:", os.getcwd())

In [1]:
import os, sys, json, warnings; warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "8")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
sys.path.insert(0, os.getcwd()); sys.path.insert(0, "monitoring"); sys.path.insert(0, "limit")
import splits as S, monitor as M
dev = S.load_dev()
for c in S.CAT: dev[c] = dev[c].astype("category")
L, W = S.LABEL, S.WEIGHT
sch = json.load(open("bundle/fraud_xgb_mask_v1_1/schema.json")); F = [f["name"] for f in sch["features"]]; CATS = [f["name"] for f in sch["features"] if f["type"] == "categorical"]; NUM = [f for f in F if f not in CATS]
model = joblib.load("models/fraud_v1_1_xgb_mask.joblib")
pool, oot = S.cv_pool(dev), S.dev_oot(dev)
score = lambda d: model.predict_proba(d[F])[:, 1]
ref = M.Reference(10).fit(pool, NUM, CATS, score=score(pool), weight=W); ref.to_json("monitoring/reference_v1_1.json")
ks_ref = M.ks_points(pool[L], score(pool), pool[W]); print(f"referensi: {len(NUM)} fitur numerik + {len(CATS)} kategorikal + skor | KS pelatihan {ks_ref:.1f} poin | tersimpan: monitoring/reference_v1_1.json")

referensi: 61 fitur numerik + 2 kategorikal + skor | KS pelatihan 98.6 poin | tersimpan: monitoring/reference_v1_1.json


## 1. Uji alarm palsu
Dua jendela yang seharusnya stabil: DEV Juli-September (periode yang tidak dipakai melatih, musiman sedikit berbeda) dan 10.000 nasabah simulasi baru (seed berbeda). Hasil yang diharapkan: skor stabil, dan paling banyak beberapa fitur pada status 'pantau'.

In [2]:
card_feats = json.load(open("limit/artifacts/scorecard.json"))["features"]
sim = pd.read_parquet("data/sim/sim_days.parquet", columns=sorted(set(F + card_feats + ["customer_id", "snap_date"]))).sample(500_000, random_state=1).reset_index(drop=True)   # sampel 500 ribu baris cukup untuk PSI dan hemat memori
for c in S.CAT:
    if c in sim: sim[c] = sim[c].astype("category")
rows = {}
for name, d, w in (("DEV Juli-September", oot, W), ("Nasabah simulasi baru", sim, None)):
    t, sp = ref.psi_table(d, score=score(d), weight=w); rows[name] = dict(psi_skor=sp, status_skor=M.status(sp), fitur_drift=int((t.status == "drift").sum()), fitur_pantau=int((t.status == "pantau").sum()), psi_fitur_tertinggi=t.psi.max(), fitur_tertinggi=t.feature.iloc[0])
    if name.startswith("DEV"): t_dev = t
pd.DataFrame(rows).T

,psi_skor,status_skor,fitur_drift,fitur_pantau,psi_fitur_tertinggi,fitur_tertinggi
DEV Juli-September,0.001625,stabil,0,0,0.780109,kyc_age_months
Nasabah simulasi baru,0.000179,stabil,0,0,0.023389,kyc_age_months


In [3]:
t_dev.head(8).round(4)

,feature,psi,missing_ref,missing_now,status,naik_kosong,alarm_kosong
0,kyc_age_months,0.7801,0.0000,0.0000,waktu,0.0000,False
1,inflow_cv_90,0.0745,0.0064,0.0269,stabil,0.0205,False
2,regular_inflow_windows_90,0.0507,0.0000,0.0000,stabil,0.0000,False
3,distinct_payee_30,0.0377,0.0000,0.0000,stabil,0.0000,False
4,occupation,0.0329,0.0000,0.0000,stabil,0.0000,False
5,min_balance_90,0.0262,0.0000,0.0000,stabil,0.0000,False
6,balance_to_income,0.0245,0.0000,0.0000,stabil,0.0000,False
7,n_out_7,0.0196,0.0000,0.0000,stabil,0.0000,False


## 2. Uji deteksi: drift yang sengaja disuntikkan
Tiga skenario pada salinan DEV Juli-September: **(a)** sumber data payee mati pada 30% baris (fitur payee kosong), **(b)** perilaku bergeser seperti musim Lebaran (porsi malam 2,5x dan nilai outflow 1,6x), **(c)** perubahan kecil (5%) yang tidak boleh memicu alarm.

In [4]:
def inject(kind):
    d = oot.copy(); r = np.random.default_rng(3)
    if kind == "a":
        pay = [c for c in F if c.startswith(("new_payee", "distinct", "payee_concentration", "top_payee", "ftp_large"))]; hit = r.random(len(d)) < 0.3
        for c in pay: d.loc[hit, c] = np.nan
    elif kind == "b":
        for c in [c for c in NUM if c.startswith("night_share")]: d[c] = (d[c] * 2.5).clip(upper=1)
        for c in [c for c in NUM if c.startswith(("out_", "avg_out", "p95_out", "max_out"))]: d[c] = d[c] * 1.6
    else:
        for c in NUM: d[c] = d[c] * 1.05
    return d
from sklearn.metrics import average_precision_score
ks_now = lambda d: M.ks_points(d[L], score(d), d[W]); ks_oot = ks_now(oot); pr_oot = average_precision_score(oot[L], score(oot), sample_weight=oot[W])
res = {}
for k, name in (("a", "(a) sumber payee mati (30% baris)"), ("b", "(b) perilaku bergeser seperti Lebaran"), ("c", "(c) perubahan kecil 5%")):
    d = inject(k); t, sp = ref.psi_table(d, score=score(d), weight=W); pr_now = average_precision_score(d[L], score(d), sample_weight=d[W]); dec = M.decide(t, sp, ks_oot - ks_now(d), pr_oot, pr_now)
    res[name] = dict(psi_skor=sp, fitur_drift=dec["features_drift"], alarm_kosong=dec["missing_alarms"], KS_turun_poin=ks_oot - ks_now(d), PR_AUC_turun_relatif=dec["prauc_rel_drop"], keputusan_proposal_saja=("RETRAIN" if (dec["score_psi_trigger"] or dec["ks_trigger"]) else "tidak ada tindakan"), keputusan_dengan_pemicu_tambahan=dec["action"])
det = pd.DataFrame(res).T; det

,psi_skor,fitur_drift,alarm_kosong,KS_turun_poin,PR_AUC_turun_relatif,keputusan_proposal_saja,keputusan_dengan_pemicu_tambahan
(a) sumber payee mati (30% baris),0.032758,11,11,1.369698,0.144824,tidak ada tindakan,INVESTIGATE DATA PIPELINE
(b) perilaku bergeser seperti Lebaran,0.233935,6,0,2.04674,0.179727,tidak ada tindakan,REVIEW
(c) perubahan kecil 5%,0.002779,1,0,-0.039314,0.011259,tidak ada tindakan,OK


**Cara membaca:** skenario (a) harus menunjuk fitur payee sebagai penyebab; (b) fitur porsi malam dan outflow; (c) tidak boleh memicu tindakan. Kolom `keputusan_proposal_saja` memakai dua pemicu di proposal (PSI skor > 0,25, KS turun > 5 poin); kolom `keputusan_dengan_pemicu_tambahan` menambah penurunan relatif PR-AUC (> 20%), alarm nilai kosong (kenaikan > 5 poin persentase), dan jumlah fitur drift (>= 5).
**Temuan:** pada skenario sumber payee mati, skor model hampir tidak bergeser karena model dilatih tahan terhadap nilai kosong, sehingga dua pemicu proposal tidak menyala sama sekali; alarm nilai kosong dan PSI per fitur-lah yang menangkapnya. Pada pergeseran perilaku ala Lebaran, PSI skor dan KS berada tepat di bawah ambang. Model yang robust menyembunyikan gangguan input dari pemantau berbasis skor, sehingga pemantau input wajib ada.

In [5]:
d = inject("a"); t, _ = ref.psi_table(d, weight=W); t["naik_kosong"] = t.missing_now - t.missing_ref
t.sort_values("naik_kosong", ascending=False).head(6)[["feature", "missing_ref", "missing_now", "naik_kosong", "psi", "status"]].round(3)

,feature,missing_ref,missing_now,naik_kosong,psi,status
0,distinct_payee_30,0.0,0.299,0.299,2.534,drift
1,payee_concentration_90,0.0,0.299,0.299,2.517,drift
2,distinct_per_tx_30,0.0,0.299,0.299,2.508,drift
3,distinct_payee_1,0.0,0.299,0.299,2.508,drift
4,top_payee_share_1,0.0,0.299,0.299,2.505,drift
5,new_payee_cnt_7,0.0,0.299,0.299,2.505,drift


## 3. Kinerja bulanan pada hasil yang matang
**Catatan penting:** Oktober-Juni adalah bulan pelatihan (dalam sampel), sehingga PR-AUC di sana (~0,92) jauh lebih tinggi daripada Juli-Agustus (di luar sampel, ~0,65-0,71). Itu **bukan degradasi**; baseline pemantauan yang adil adalah performa di luar sampel. KS nyaris tidak bergerak antara keduanya (98-99 vs 96), padahal PR-AUC turun sepertiga: pada model kejadian langka, KS adalah penanda yang tumpul, sehingga pemicu berbasis KS saja tidak cukup.

KS (dalam poin) dan PR-AUC per bulan pada seluruh DEV, dibanding KS pelatihan. Pemicu proposal: KS turun lebih dari 5 poin. Pada data sintetis ini label hari fraud langsung matang; di produksi, label baru matang setelah laporan atau konfirmasi (jeda beberapa hari sampai minggu), sehingga bulan terakhir ditandai belum matang.

In [6]:
from sklearn.metrics import average_precision_score
alld = pd.concat([pool, oot], ignore_index=True); alld["bulan"] = alld.snap_date.dt.to_period("M"); alld["s"] = score(alld)
mon = alld.groupby("bulan").apply(lambda g: pd.Series(dict(baris=len(g), hari_fraud=int(g[L].sum()), KS=M.ks_points(g[L], g.s, g[W]) if g[L].sum() > 20 else np.nan,
                                                          PR_AUC=average_precision_score(g[L], g.s, sample_weight=g[W]) if g[L].sum() > 20 else np.nan)))
mon["KS_turun_poin"] = ks_ref - mon.KS; mon["pemicu_KS"] = mon.KS_turun_poin > M.KS_DROP_POINTS
pr_val = mon.loc[[p for p in mon.index if str(p) in ("2026-07", "2026-08")], "PR_AUC"].mean()                      # baseline yang adil = performa di luar sampel (Juli-Agustus), bukan bulan pelatihan
mon["PR_AUC_turun_vs_validasi"] = (pr_val - mon.PR_AUC) / pr_val
mon["catatan"] = np.where(mon.hari_fraud <= 100, "sampel kecil / belum matang", np.where(mon.index < pd.Period("2026-07"), "bulan pelatihan (dalam sampel)", "di luar sampel"))
mon.round(3)

,baris,hari_fraud,KS,PR_AUC,KS_turun_poin,pemicu_KS,PR_AUC_turun_vs_validasi,catatan
bulan,,,,,,,,
2025-10,40458.0,250.0,99.200,0.919,-0.580,False,-0.358,bulan pelatihan (dalam sampel)
2025-11,39302.0,472.0,99.044,0.908,-0.424,False,-0.342,bulan pelatihan (dalam sampel)
2025-12,40691.0,634.0,98.908,0.916,-0.288,False,-0.353,bulan pelatihan (dalam sampel)
2026-01,41110.0,520.0,98.918,0.931,-0.298,False,-0.375,bulan pelatihan (dalam sampel)
2026-02,37141.0,443.0,98.734,0.915,-0.114,False,-0.351,bulan pelatihan (dalam sampel)
2026-03,40369.0,545.0,98.355,0.923,0.265,False,-0.363,bulan pelatihan (dalam sampel)
2026-04,39233.0,503.0,98.965,0.932,-0.345,False,-0.377,bulan pelatihan (dalam sampel)
2026-05,40653.0,431.0,98.011,0.917,0.609,False,-0.355,bulan pelatihan (dalam sampel)
2026-06,40051.0,324.0,98.868,0.915,-0.248,False,-0.352,bulan pelatihan (dalam sampel)


## 4. Fairness per kuartal (kesamaan peluang)
Recall pada ambang alert 0,1% traffic per gender, usia, dan wilayah, dengan rasio terkecil/terbesar untuk kelompok yang punya cukup hari fraud (>= 30 per kuartal). Gender dan usia tidak menjadi fitur model, tetapi tetap diperiksa hasilnya.

In [7]:
alld["kuartal"] = alld.snap_date.dt.to_period("Q"); out = []
for q, g in alld.groupby("kuartal"):
    thr = np.sort(g.s.values)[::-1][np.searchsorted(np.cumsum(g[W].values[np.argsort(-g.s.values)]), 0.001 * g[W].sum())]; g = g.assign(f=g.s >= thr)
    for col, lab in (("gender", "gender"), ("age_ord", "usia"), ("region", "wilayah")):
        r_ = g[g[L] == 1].groupby(col).f.agg(["mean", "size"]); r_ = r_[r_["size"] >= 30]["mean"]
        gap = M.group_gap(r_, min_ratio=0.8); out.append(dict(kuartal=str(q), atribut=lab, kelompok_memadai=len(r_), recall_min=r_.min(), recall_max=r_.max(), min_per_max=gap["min_over_max"], di_bawah_batas_080=gap["breach"]))
pd.DataFrame(out).round(3)

,kuartal,atribut,kelompok_memadai,recall_min,recall_max,min_per_max,di_bawah_batas_080
0,2025Q4,gender,2,0.937,0.955,0.981,False
1,2025Q4,usia,5,0.912,0.969,0.941,False
2,2025Q4,wilayah,10,0.787,0.978,0.805,False
3,2026Q1,gender,2,0.947,0.951,0.996,False
4,2026Q1,usia,5,0.882,0.976,0.903,False
5,2026Q1,wilayah,11,0.926,0.992,0.934,False
6,2026Q2,gender,2,0.955,0.955,1.000,False
7,2026Q2,usia,5,0.940,0.986,0.953,False
8,2026Q2,wilayah,10,0.922,0.987,0.934,False
9,2026Q3,gender,2,0.839,0.853,0.983,False


## 5. Tier migration, churn, dan lantai baseline
Dari simulasi 10.000 nasabah (notebook 05, model fraud v1.1): pangsa tier per bulan, matriks migrasi per kuartal, tingkat perubahan tier bulanan, dan pemeriksaan lantai non-negosiasi (tidak ada nasabah di bawah tier baseline).

In [8]:
import gc
for v in ("alld", "pool", "oot", "dev", "t", "d"): globals().pop(v, None)      # bebaskan memori sebelum simulasi
gc.collect()
import simulate as SM
prep = SM.prepare(None, 1, "data/sim"); out_, mig_, ctx = SM.run(prep, seed=1)
th, base = ctx["tier_hist"], ctx["baseline"]; days = pd.date_range(SM.WIN_START, periods=SM.D)
share = pd.DataFrame({f"T{k}": (th[:, SM.T0:] == k).mean(axis=0) for k in range(1, 6)}, index=days[SM.T0:]).resample("MS").mean()
chg = pd.Series((np.diff(th[:, SM.T0 - 1:], axis=1) != 0).sum(axis=0) / th.shape[0], index=days[SM.T0:]).resample("MS").sum()
print("Pangsa tier rata-rata per bulan:"); display(share.round(3))
print("Perubahan tier per nasabah per bulan:", chg.round(3).to_dict() if len(chg) < 3 else "")
chg.round(3).to_frame("perubahan_per_nasabah")

Pangsa tier rata-rata per bulan:


,T1,T2,T3,T4,T5
2025-10-01,0.347,0.377,0.178,0.080,0.018
2025-11-01,0.326,0.385,0.186,0.084,0.018
2025-12-01,0.315,0.391,0.189,0.086,0.018
2026-01-01,0.310,0.397,0.188,0.087,0.018
2026-02-01,0.311,0.400,0.184,0.087,0.018
2026-03-01,0.310,0.400,0.184,0.087,0.018
2026-04-01,0.310,0.390,0.189,0.093,0.018
2026-05-01,0.309,0.380,0.197,0.096,0.018
2026-06-01,0.308,0.374,0.201,0.098,0.018
2026-07-01,0.307,0.382,0.193,0.099,0.018


Perubahan tier per nasabah per bulan: 


,perubahan_per_nasabah
2025-10-01,0.175
2025-11-01,0.076
2025-12-01,0.047
2026-01-01,0.051
2026-02-01,0.039
2026-03-01,0.040
2026-04-01,0.057
2026-05-01,0.055
2026-06-01,0.058
2026-07-01,0.056


In [9]:
q_end = [pd.Timestamp(x) for x in ("2025-12-31", "2026-03-31", "2026-06-30", "2026-09-30")]; start = pd.Timestamp("2025-10-01")
for a, b in zip([start] + q_end[:-1], q_end):
    ia, ib = (a - SM.WIN_START).days, (b - SM.WIN_START).days
    print(f"Migrasi tier {a.date()} -> {b.date()}"); display(M.migration_matrix(th[:, ia], th[:, ib]))
print("nasabah di bawah tier baseline pada hari mana pun (harus 0):", int((th[:, SM.T0:] < base[:, None]).any(axis=1).sum()))

Migrasi tier 2025-10-01 -> 2025-12-31


akhir,1,2,3,4,5
awal,,,,,
1,3145,976,0,0,0
2,0,2942,659,0,0
3,0,0,1233,203,0
4,0,0,0,657,0
5,0,0,0,0,185


Migrasi tier 2025-12-31 -> 2026-03-31


akhir,1,2,3,4,5
awal,,,,,
1,2958,187,0,0,0
2,132,3597,189,0,0
3,0,213,1618,61,0
4,0,0,38,822,0
5,0,0,0,0,185


Migrasi tier 2026-03-31 -> 2026-06-30


akhir,1,2,3,4,5
awal,,,,,
1,2945,145,0,0,0
2,134,3506,357,0,0
3,0,146,1558,141,0
4,0,0,44,839,0
5,0,0,0,0,185


Migrasi tier 2026-06-30 -> 2026-09-30


akhir,1,2,3,4,5
awal,,,,,
1,2935,144,0,0,0
2,105,3455,237,0,0
3,14,286,1583,76,0
4,0,6,45,929,0
5,0,0,0,0,185


nasabah di bawah tier baseline pada hari mana pun (harus 0): 0


## 6. Distribusi reason code scorecard
Reason code (alasan skor rendah) dihitung tiap bulan pada sampel nasabah simulasi; PSI dihitung terhadap bulan pertama. Pergeseran besar berarti populasi atau perilaku berubah, atau fitur scorecard rusak.

In [10]:
from scorecard import Scorecard
card = Scorecard.from_json("limit/artifacts/scorecard.json"); sim["bulan"] = sim.snap_date.dt.to_period("M"); top = {}
for m_, g in sim.groupby("bulan"):
    g = g.sample(min(1500, len(g)), random_state=1); rc = card.reason_codes(g, 1); top[str(m_)] = pd.Series([r[0].split(" (")[0] if r else "—" for r in rc]).value_counts(normalize=True)
rcd = pd.DataFrame(top).fillna(0); first = rcd.iloc[:, 0].values
psi_rc = {c: M.psi_value(first, rcd[c].values) for c in rcd.columns}; display(rcd.round(3).head(6))
pd.Series(psi_rc, name="PSI reason code vs bulan pertama").round(4).to_frame().T

,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
declared income,0.794,0.777,0.771,0.783,0.779,0.807,0.812,0.813,0.784,0.777,0.751,0.756
many different payees per transaction,0.001,0.004,0.003,0.001,0.001,0.003,0.002,0.007,0.004,0.001,0.004,0.000
pass-through of incoming funds,0.000,0.000,0.000,0.000,0.001,0.001,0.000,0.001,0.001,0.000,0.001,0.002
unstable monthly inflow,0.001,0.001,0.000,0.001,0.000,0.001,0.002,0.003,0.003,0.001,0.008,0.003
unusual payee concentration,0.204,0.219,0.225,0.215,0.219,0.189,0.184,0.176,0.208,0.221,0.236,0.239


,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
PSI reason code vs bulan pertama,0.0,0.0078,0.0103,0.0012,0.0077,0.0052,0.0042,0.0213,0.0091,0.0022,0.026,0.0152


## 7. Ringkasan pemantau dan ambangnya

In [11]:
summary = pd.DataFrame([
  dict(pemantau="PSI skor model", ambang="> 0,25 retrain", status_data_stabil=f"{rows['DEV Juli-September']['status_skor']} ({rows['DEV Juli-September']['psi_skor']:.3f})", terdeteksi_saat_drift="skenario perilaku bergeser: " + str(round(res['(b) perilaku bergeser seperti Lebaran']['psi_skor'], 3))),
  dict(pemantau="PSI per fitur dan tingkat kosong", ambang="> 0,25 drift; kenaikan kosong", status_data_stabil=f"{rows['DEV Juli-September']['fitur_drift']} fitur drift", terdeteksi_saat_drift="sumber payee mati terlihat dari kenaikan tingkat kosong"),
  dict(pemantau="KS bulanan pada label matang", ambang="turun > 5 poin retrain", status_data_stabil=f"bulan dengan pemicu: {int(mon.pemicu_KS.sum())}", terdeteksi_saat_drift="dihitung pada data yang dirusak"),
  dict(pemantau="Fairness recall (kuartal)", ambang="min/max < 0,80 selidiki", status_data_stabil="lihat bagian 4", terdeteksi_saat_drift="-"),
  dict(pemantau="Tier migration dan churn", ambang="tier < baseline harus 0", status_data_stabil="0 pelanggaran", terdeteksi_saat_drift="-"),
  dict(pemantau="Distribusi reason code", ambang="PSI > 0,25 selidiki", status_data_stabil=f"PSI maksimum {max(psi_rc.values()):.3f}", terdeteksi_saat_drift="-")])
json.dump(dict(psi_stable=rows, injected=res, monthly=mon.round(4).astype(str).rename(index=str).to_dict(), reason_psi=psi_rc), open("monitoring/monitoring_report.json", "w"), indent=1, default=str)
summary

,pemantau,ambang,status_data_stabil,terdeteksi_saat_drift
0,PSI skor model,"> 0,25 retrain",stabil (0.002),skenario perilaku bergeser: 0.234
1,PSI per fitur dan tingkat kosong,"> 0,25 drift; kenaikan kosong",0 fitur drift,sumber payee mati terlihat dari kenaikan tingk...
2,KS bulanan pada label matang,turun > 5 poin retrain,bulan dengan pemicu: 0,dihitung pada data yang dirusak
3,Fairness recall (kuartal),"min/max < 0,80 selidiki",lihat bagian 4,-
4,Tier migration dan churn,tier < baseline harus 0,0 pelanggaran,-
5,Distribusi reason code,"PSI > 0,25 selidiki",PSI maksimum 0.026,-
